# Reward-gap follow-up: checkpoint review and iterative kNN PPO

**Run this one notebook on the RTX PRO 6000 (95–96 GiB).** Extract the entire ZIP, open this notebook inside `reward_gap_followup`, start a fresh Python 3.12 kernel, and choose **Run → Run All Cells**.

This runs the saved-checkpoint **128/256-token evaluation**, offline controls, and a **two-round static-versus-iterative kNN PPO experiment**. Teacher feedback adds signed gaps to frozen-proxy vectors between rounds. Static and iterative conditions share the exact round-one policy, value head and optimizer. A capped correction is a separate ablation.

All saved adapters, memory, data and requirements are included. Large model weights use your existing Hugging Face cache first. No login cell or hidden token prompt is used. If files are missing and downloads are enabled, a genuine server rate limit can still block downloading.

**The study is resumable and runs in the background after preflight.** Closing the notebook is okay; the pod must stay running. The default study is substantial (1,800 PPO rollout/update iterations and about 76k teacher-scored answers); it may take longer than a night. This package contains no new measured GPU outcomes. Human review is manual and remains pending until you import ratings.

See `README_RUN.md` and `PROTOCOL.md` for the complete design.

## 1. Configuration

Keep the defaults for the full study. `RUN_NEW_PPO=False` completes saved-checkpoint evaluation and offline diagnostics first; you can later set it to `True` and resume. `EXTRA_HF_CACHE` should point to a Hugging Face **hub cache directory**, if the models are in a nonstandard location. Do not paste tokens into saved notebook source.

In [1]:
from pathlib import Path
import sys
import subprocess
import json

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "PACKAGE_MANIFEST.json").is_file():
    candidate = PROJECT_ROOT / "reward_gap_followup"
    if not (candidate / "PACKAGE_MANIFEST.json").is_file():
        raise FileNotFoundError("Extract the whole ZIP and open RUN_FOLLOWUP.ipynb inside reward_gap_followup.")
    PROJECT_ROOT = candidate
sys.path.insert(0, str(PROJECT_ROOT))

RUN_NEW_PPO = True
ALLOW_DOWNLOADS = True
EXTRA_HF_CACHE = None   # Example: "/workspace/huggingface/hub"
REPAIR_CUDA = False    # Only enable if the checker reports an incompatible CUDA build.

from common import read_json, write_json
from launch import active_job
config_path = PROJECT_ROOT / "config.json"
config = read_json(config_path)
updated = {**config, "run_new_ppo": RUN_NEW_PPO,
           "allow_downloads": ALLOW_DOWNLOADS, "extra_hf_cache": EXTRA_HF_CACHE}
if updated != config:
    if active_job():
        raise RuntimeError("A study is running. Pause it before changing execution settings.")
    write_json(config_path, updated)
print("Project:", PROJECT_ROOT)
print("Mode:", "Full two-round study" if RUN_NEW_PPO else "Checkpoint review + offline diagnostics only")
print("Python:", sys.executable)

Project: /workspace/reward_gap_followup
Mode: Full two-round study
Python: /usr/local/bin/python


## 2. Start or resume

This cell streams the requirements checks, CPU tests and GPU preflight. Preflight loads the pinned models, checks saved-memory and teacher scores, loads a saved adapter, and performs a small discarded PPO update. **The study launches only if these checks pass.**

Once `Started PID ...` appears, this cell returns and the worker continues separately. Re-running it resumes compatible saved work or reports that the study is already running. It does not start a duplicate worker. If setup or preflight fails, read the error here; the background study has not started.

In [2]:
command = [sys.executable, "-u", str(PROJECT_ROOT / "launch.py"), "start"]
if REPAIR_CUDA:
    command.append("--repair-cuda")
with subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
if return_code:
    raise RuntimeError(f"Launcher stopped with code {return_code}. Read the error above; completed artifacts are preserved.")

Dependencies: 2.8.0+cu128 5.17.0 0.20.0
CUDA available: True build: 12.8
Dependency check finished. GPU execution is checked by preflight.
Offline correctness tests passed.
Running GPU preflight now; the detached study starts only after it passes.
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition VRAM GiB: 95.0
All three pinned models are ready. Prompt data and adapters are bundled.

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 454.88it/s]

Loading weights: 100%|██████████| 399/399 [00:01<00:00, 218.49it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 818.55it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 13345.35it/s]
PREFLIGHT PASSED: GPU, saved adapter, whole-answer scoring, memory/judge parity and PPO gradients.
Started PID 1507. You can close the notebook/browser; keep the RunPod running.
Status: python launch.py status
Pause safely: python launch.py stop
Resume: python launch.py start
Log: /workspace/reward_gap_followup/outputs/training.l

## 3. Live status and files — re-run this cell whenever you want

`Process: running` is checked against the operating-system process. `stage=complete` means the GPU study finished; it does not mean humans have rated the answers. `evaluation_complete` means you chose the shorter evaluation-only mode. A process that is **not running** with `failed` or an unfinished stage needs attention; inspect the log.

The result ZIP is created at completion or a cooperative pause/failure. Reports appear after completed evaluations. Only the ZIPs labeled **BLINDED** should be sent to reviewers; the full results contain the private mapping.

In [6]:
from IPython.display import display, FileLink

def show_status_and_links():
    subprocess.run([sys.executable, str(PROJECT_ROOT / "launch.py"), "status"],
                   cwd=PROJECT_ROOT, check=True)
    latest = PROJECT_ROOT / "outputs/latest.json"
    if not latest.is_file():
        return
    output = PROJECT_ROOT / read_json(latest)["relative_output"]
    candidates = [output / "reports/report.html", output / "important_outcomes_followup.zip"]
    candidates += sorted((output / "review").glob("*/*_BLINDED.zip"))
    for path in candidates:
        if path.is_file():
            try:
                link_path = path.relative_to(Path.cwd())
            except ValueError:
                link_path = path
            display(FileLink(str(link_path)))

show_status_and_links()

Process: not running
{
  "stage": "complete",
  "updated_unix": 1789052758.359228,
  "elapsed_seconds": 26200.93693413213,
  "final_policy_runs": 12,
  "note": "GPU study complete. Blinded human ratings are a separate manual step."
}

Results folder: /workspace/reward_gap_followup/outputs/study_cfdbaf579047d418
Report: /workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/reports/report.html
Results ZIP: /workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/important_outcomes_followup.zip

Recent log:
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 784/1024
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 800/1024
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 816/1024
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 832/1024
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 848/1024
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 864/1024
legacy_fresh legacy_knn_signed_s44 fresh_final cap=256: 880/1024
legacy_fresh legacy_kn

/workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/reports/report.html

/workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/important_outcomes_followup.zip

/workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/review/checkpoint_review/checkpoint_review_BLINDED.zip

/workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/review/iterative_vs_static_review/iterative_vs_static_review_BLINDED.zip

/workspace/reward_gap_followup/outputs/study_cfdbaf579047d418/review/new_policy_vs_raw_review/new_policy_vs_raw_review_BLINDED.zip

## 4. Optional: import completed human-review ratings

This cell does nothing during Run All unless you provide rating files. Extract a BLINDED ZIP and open its `REVIEW.html` in a browser. Reviewers independently rate random pairs for usefulness, correctness, inappropriate refusal, completeness and preference, then export JSON.

Upload the exported JSON files to this project. Set the correct `REVIEW_NAME` and list **all intended current exports for that pack** below. Run this cell after the worker has finished or paused. It validates and unblinds the ratings, rebuilds the report and refreshes the results ZIP. Partial review coverage is shown explicitly.

In [4]:
REVIEW_NAME = "iterative_vs_static_review"
RATINGS_FILES = []  # Example: ["ratings_reviewer1_....json", "ratings_reviewer2_....json"]

if RATINGS_FILES:
    if active_job():
        raise RuntimeError("Wait for the worker to finish or pause before importing ratings and rebuilding reports.")
    output = PROJECT_ROOT / read_json(PROJECT_ROOT / "outputs/latest.json")["relative_output"]
    folder = output / "review" / REVIEW_NAME
    if not (folder / "private/key.json").is_file():
        raise FileNotFoundError("This review pack has not been created yet. Check REVIEW_NAME and study status.")
    paths = [str((PROJECT_ROOT / name).resolve()) for name in RATINGS_FILES]
    subprocess.run([sys.executable, str(PROJECT_ROOT / "review.py"),
                    "--folder", str(folder), "--ratings", *paths], cwd=PROJECT_ROOT, check=True)
    subprocess.run([sys.executable, str(PROJECT_ROOT / "reporting.py"),
                    "--output", str(output), "--export"], cwd=PROJECT_ROOT, check=True)
    show_status_and_links()
else:
    print("No ratings imported. Human review remains a separate manual step.")

No ratings imported. Human review remains a separate manual step.


## 5. Pause only when you choose

Leave `PAUSE_NOW=False` during Run All. To pause, set it to `True` and run this cell. **Wait for `stage=paused` and no running process before stopping the pod.** The current update or evaluation batch may take several minutes. Preserve the entire project directory to resume on another pod; the results ZIP omits optimizer checkpoints.

To resume, reset `PAUSE_NOW=False` and run section 2. To inspect progress without changing anything, run section 3.

In [5]:
PAUSE_NOW = False
if PAUSE_NOW:
    subprocess.run([sys.executable, str(PROJECT_ROOT / "launch.py"), "stop"],
                   cwd=PROJECT_ROOT, check=True)
else:
    print("No pause requested. Use section 3 to check whether the worker is running.")

No pause requested. Use section 3 to check whether the worker is running.
